In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# 다양성
# 관련성
# 균형
# https://arxiv.org/pdf/2104.08786 프롬프트 순서에 따른 내용을 넣어줌
# https://arxiv.org/pdf/2102.09690 few-shot 예시를 제시할 때
# https://arxiv.org/pdf/2101.06804 예시를 전달할 때, KNN(비슷한 애들끼리 묶음) 알고리즘을 사용

In [2]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate, ChatPromptTemplate, FewShotChatMessagePromptTemplate

In [3]:
examples = [
    {'input' : '이 제품 정말 최고입니다!', 'output' : '긍정'},
    {'input' : '품질이 너무 안 좋아요', 'output' : '부점'},
    {'input' : '보통이에요, 무난합니다.', 'output' : '중립'},
    {'input' : '디자인은 좋은데 성능이 아쉬워요', 'output' : '혼합'}
]

In [6]:
example_template = PromptTemplate(
    input_variables=['input', 'output'],
    template='리뷰 : {input}\n감정 : {output}'
)

fewshot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix='다음 예시를 참고해서 리뷰의 감정을 분류해주세요.\n',
    suffix='\n리뷰 : {input}\n감정 :',
    input_variables=['input'],
    example_separator='\n\n'
)

In [7]:
formatted = fewshot_prompt.format(input='가격은 비싸지만 품질이 뛰어나요')
print(formatted)

다음 예시를 참고해서 리뷰의 감정을 분류해주세요.


리뷰 : 이 제품 정말 최고입니다!
감정 : 긍정

리뷰 : 품질이 너무 안 좋아요
감정 : 부점

리뷰 : 보통이에요, 무난합니다.
감정 : 중립

리뷰 : 디자인은 좋은데 성능이 아쉬워요
감정 : 혼합


리뷰 : 가격은 비싸지만 품질이 뛰어나요
감정 :


In [9]:
example_prompt_chat = ChatPromptTemplate.from_messages([
    ('human', '리뷰 : {input}'),
    ('ai', '감정 : {output}')
])

fewshot_chat_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt_chat
)

In [10]:
final_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 감정 분석 전문가입니다. 리뷰의 감정을 긍정/부정/중립/혼합 중 하나로 분류해주세요'),
    fewshot_chat_prompt,
    ('human', '리뷰 : {input}')
])

In [12]:
chain = final_prompt | llm
result = chain.invoke({'input' : '가격은 비싸지만 품질이 뛰어나요'})
print(result.content)

감정 : 혼합


In [13]:
large_examples = [
    {"input": "정말 만족합니다. 강력 추천!", "output": "긍정"},
    {"input": "배송이 빠르고 포장이 꼼꼼해요.", "output": "긍정"},
    {"input": "가격 대비 훌륭합니다.", "output": "긍정"},
    {"input": "품질이 최악입니다. 환불 요청했어요.", "output": "부정"},
    {"input": "고객센터 응대가 너무 불친절합니다.", "output": "부정"},
    {"input": "택배가 분실되었어요.", "output": "부정"},
    {"input": "보통이에요. 특별한 점은 없습니다.", "output": "중립"},
    {"input": "사진과 동일한 제품입니다.", "output": "중립"},
    {"input": "디자인은 예쁜데 내구성이 약해요.", "output": "혼합"},
    {"input": "기능은 많은데 사용법이 복잡합니다.", "output": "혼합"},
]

In [14]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

In [15]:
from langchain_core.vectorstores import InMemoryVectorStore

In [16]:
example_selector = SemanticSimilarityExampleSelector.from_examples(
    large_examples,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    InMemoryVectorStore,
    k=3
)

In [17]:
queries = [
    '배송이 너무 느려요',
    '값은 좀 나가지만 성능은 훌륭해요',
    '무난한 제품이에요'
]

for q in queries:
    selected = example_selector.select_examples({'input':q})
    print(f"query : {q}")
    for s in selected:
        print(f"selected : {s['output']}, {s['input']}")

query : 배송이 너무 느려요
selected : 긍정, 배송이 빠르고 포장이 꼼꼼해요.
selected : 부정, 고객센터 응대가 너무 불친절합니다.
selected : 부정, 택배가 분실되었어요.
query : 값은 좀 나가지만 성능은 훌륭해요
selected : 긍정, 가격 대비 훌륭합니다.
selected : 혼합, 기능은 많은데 사용법이 복잡합니다.
selected : 혼합, 디자인은 예쁜데 내구성이 약해요.
query : 무난한 제품이에요
selected : 중립, 사진과 동일한 제품입니다.
selected : 부정, 품질이 최악입니다. 환불 요청했어요.
selected : 긍정, 가격 대비 훌륭합니다.


In [19]:
# 동적으로 선택 -> FewShotChatMessagePromptTemplate -> llm 답변
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "리뷰: {input}"),
    ("ai", "감정: {output}")
])

example_selector = SemanticSimilarityExampleSelector.from_examples(
    large_examples,
    OpenAIEmbeddings(),
    InMemoryVectorStore,
    k=3
)

fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "리뷰의 감정을 분석하세요 (긍정/부정)"),
    fewshot_prompt,
    ("human", "리뷰: {input}")
])

chain = final_prompt | llm

response = chain.invoke({
    "input": "이거 진짜 별로임"
})

print(response.content)

감정: 부정


In [20]:
# 프롬프트 성능을 측정하기 위한 내용
eval_dataset = [
    {"text": "완벽한 제품입니다! 강력 추천합니다.", "expected": "긍정"},
    {"text": "두 번 다시 구매하지 않겠습니다.", "expected": "부정"},
    {"text": "평범합니다. 특별히 좋지도 나쁘지도 않아요.", "expected": "중립"},
    {"text": "기능은 좋은데 AS가 아쉬워요.", "expected": "혼합"},
    {"text": "가격도 착하고 품질도 좋아요.", "expected": "긍정"},
    {"text": "사진과 너무 달라서 실망했습니다.", "expected": "부정"},
    {"text": "그냥 쓸만 합니다.", "expected": "중립"},
    {"text": "배송은 빠른데 제품이 기대 이하에요.", "expected": "혼합"},
]

In [25]:
def evaluate_zero_shot(dataset):
    results = []
    for item in dataset:
        response = llm.invoke([
            SystemMessage(content = "리뷰의 감정을 분류하세요. 반드시 '긍정', '부정', '중립', '혼합' 중 하나만 출력하세요"),
            HumanMessage(content = item['text'])
        ]).content
        
        predicted = response.strip()
        results.append({
            'text' : item['text'],
            'expected' : item['expected'],
            'predicted' : predicted,
            'correct' : predicted == item['expected'],
        })
        
    return results

In [26]:
def evaluate_few_shot(dataset, examples):
    results = []
    base_messages = [SystemMessage(content = "리뷰의 감정을 분류하세요. 반드시 '긍정', '부정', '중립', '혼합' 중 하나만 출력하세요")]
    for ex in examples:
        base_messages.append(HumanMessage(content = ex['input']))
        base_messages.append(AIMessage(content = ex['output']))
        
    for item in dataset:
        messages = base_messages + [HumanMessage(content = item['text'])]
        response = llm.invoke(messages).content
                
        predicted = response.strip()
        results.append({
            'text' : item['text'],
            'expected' : item['expected'],
            'predicted' : predicted,
            'correct' : predicted == item['expected'],
        })
        
    return results

In [27]:
few_shot_examples = [
    {"input": "정말 만족합니다. 강력 추천!", "output": "긍정"},
    {"input": "배송이 빠르고 포장이 꼼꼼해요.", "output": "긍정"},
    {"input": "가격 대비 훌륭합니다.", "output": "긍정"},
    {"input": "품질이 최악입니다. 환불 요청했어요.", "output": "부정"},
    {"input": "고객센터 응대가 너무 불친절합니다.", "output": "부정"},
]

zero_results = evaluate_zero_shot(eval_dataset)

In [28]:
fewshot_result = evaluate_few_shot(eval_dataset, few_shot_examples)

In [29]:
import pandas as pd

In [30]:
pd.DataFrame(zero_results)

,text,expected,predicted,correct
0,완벽한 제품입니다! 강력 추천합니다.,긍정,긍정,True
1,두 번 다시 구매하지 않겠습니다.,부정,부정,True
2,평범합니다. 특별히 좋지도 나쁘지도 않아요.,중립,중립,True
3,기능은 좋은데 AS가 아쉬워요.,혼합,혼합,True
4,가격도 착하고 품질도 좋아요.,긍정,긍정,True
5,사진과 너무 달라서 실망했습니다.,부정,부정,True
6,그냥 쓸만 합니다.,중립,중립,True
7,배송은 빠른데 제품이 기대 이하에요.,혼합,혼합,True


In [31]:
pd.DataFrame(fewshot_result)

,text,expected,predicted,correct
0,완벽한 제품입니다! 강력 추천합니다.,긍정,긍정,True
1,두 번 다시 구매하지 않겠습니다.,부정,부정,True
2,평범합니다. 특별히 좋지도 나쁘지도 않아요.,중립,중립,True
3,기능은 좋은데 AS가 아쉬워요.,혼합,혼합,True
4,가격도 착하고 품질도 좋아요.,긍정,긍정,True
5,사진과 너무 달라서 실망했습니다.,부정,부정,True
6,그냥 쓸만 합니다.,중립,중립,True
7,배송은 빠른데 제품이 기대 이하에요.,혼합,혼합,True


In [ ]:
# 엑셀 표에 담김
pd.DataFrame(fewshot_result).to_csv('a.csv')

In [ ]:
regression related metric : rmse(root mean squared error)

In [33]:
print(llm.invoke('LOLLAPALOOZA 라는 단어에서 L이 몇 번 나타나나요?').content)

"LOLLAPALOOZA"라는 단어에서 L은 3번 나타납니다.


In [ ]:
# CoT(Chain of Thougt) https://arxiv.org/pdf/2205.11916
# 프롬프트를 작성할 때 마지막에 한줄만 추가 해봐라 "단계별로 생각해보세요"
# 지문의 조건을 전부 나열한 후 각가의 조건들을 검토하면서 추론해줘/풀어줘
# 이 문제가 기존 유명한 문제와 어떻게 다른지를 분석하고 답해줘

In [ ]:
# few-shot CoT : CoT + 예시를 같이 줌

In [36]:
system_prompt = """
당신은 퍼즐 전문가 입니다. 아래 문제들은 유명한 문제의 **변형**입니다.
원래 문제의 답을 그래도 적용하지 마세요. 이 문제의 조건을 정확히 읽고 답하세요.

예시:
문제: 셔츠 1장을 말리는데 4시간이 걸립니다. 셔츠 5장을 말리면 몇시간?
[흔한 실수] 4 x 5 = 20시간
[올바른 생각] 동시에 널면 됩니다. 답 : 4시간

이 처럼 문제의 조건을 주의 깊게 읽고 단계별로 생각하세요
"""

In [38]:
system_prompt = """
당신은 퍼즐 전문가입니다.

이 문제들은 유명한 문제의 변형입니다.
절대로 기존에 알고 있는 답을 그대로 적용하지 말고,
문제의 조건을 정확히 분석해서 답을 도출하세요.

다음 절차를 따르세요:
1. 핵심 조건을 정리
2. 수식으로 변환
3. 논리적으로 계산
4. 결과 검증

주의:
- 평균 속도는 전체 거리 / 전체 시간으로 계산해야 합니다.
- 직관적인 평균 계산을 사용하면 안 됩니다.

최종 답만 간결하게 출력하세요.
"""

question = """
친구 집까지 평균 시속 3마일로 걸어갔습니다.
왕복 전체 평균 속도를 시속 6마일로 만들려면,
돌아올 때 얼마나 빨리 달려야 할까요?
"""

response = llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
])

print(response.content)

1. 핵심 조건 정리:
   - 거리: 친구 집까지의 거리(D) (왕복 거리 = 2D)
   - 가는 속도: 3마일/시 (t1 = D/3)
   - 돌아오는 속도: x 마일/시 (t2 = D/x)
   - 전체 평균 속도: 6마일/시

2. 수식으로 변환:
   - 전체 거리: 2D
   - 전체 시간: t1 + t2 = D/3 + D/x
   - 전체 평균 속도: (2D) / (D/3 + D/x) = 6

3. 논리적으로 계산:
   - 평균 속도 수식에 대입:
     \( \frac{2D}{\frac{D}{3} + \frac{D}{x}} = 6 \)
   - D를 양쪽에서 약분:
     \( \frac{2}{\frac{1}{3} + \frac{1}{x}} = 6 \)
   - 분모를 통분:
     \( \frac{2}{\frac{x+3}{3x}} = 6 \)
   - 분수를 뒤집어서 정리:
     \( 2 \cdot \frac{3x}{x + 3} = 6 \)
     \( \frac{6x}{x + 3} = 6 \)
   - 양변에 (x + 3)를 곱해서 정리:
     \( 6x = 6(x + 3) \)
     \( 6x = 6x + 18 \)
     \( 0 = 18 \) (모순 발생, 다시 점검)
      
   - 올바르게 정리하면:
     \( 2 = 6 \cdot \frac{x + 3}{3x} \)
     \( 6(x + 3) = 6x \)
     \( 2x + 6 = 6x \)
     \( 4x = 6 \)
     \( x = \frac{6}{4} \)
     \( x = 1.5 \) 마일/시

4. 결과 검증:
   - 가는 시간: D/3
   - 오는 시간: D/1.5 = 2D/3
   - 총 시간: D/3 + 2D/3 = D
   - 평균 속도: (2D) / D = 6 마일/시 (일치)

최종 결과: 12 마일/시
